In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn 

from src.config import TARGET, TIME_COL, UID_COL, SEED, DAGSHUB_USERNAME,DAGSHUB_REPO 
from src.data.DLRM_preprocessing import DLRMPreprocessor
from src.models.dlrm_trainer import train_dlrm_seed_ensemble

from google.colab import drive
import os
import mlflow
import mlflow.pytorch
import dagshub
from dotenv import load_dotenv

ModuleNotFoundError: No module named 'src'

In [2]:
%pip install -q mlflow dagshub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Mount Google Drive into the remote Colab container
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/IEEE-fraud-detection"
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Load credentials directly from Google Drive .env file
ENV_PATH = os.path.join(SAVE_DIR, ".env")
load_dotenv(ENV_PATH)

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)

#mlflow.set_experiment("IEEE-fraud-multimodal-nn")
print(f"MLflow tracking set to DagsHub")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=74ce4605-07e0-4169-a894-6c66ad1d5c36&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=45f3324ff41eba566e652dd10c336dff8b35114492c873bb3bb6e82d91d8801d




Accessing as anamish05

Initialized MLflow to track repo "anamish05/ieee-fraud-detection"

Repository anamish05/ieee-fraud-detection initialized!

2026/10/01 11:12:56 INFO mlflow.tracking.fluent: Experiment with name 'IEEE-fraud-multimodal-nn' does not exist. Creating a new experiment.


MLflow tracking set to DagsHub


In [7]:
from pathlib import Path

# Path inside mounted Google Drive
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/IEEE-fraud-detection")
PROCESSED_PATH = DRIVE_PROJECT_ROOT / "data/processed/features_static_after_split_TE.parquet"

# Verify file exists before loading
assert PROCESSED_PATH.exists(), f"File not found at: {PROCESSED_PATH}"

df = pd.read_parquet(PROCESSED_PATH)
print(f"Loaded engineered dataset: {df.shape}")

Loaded engineered dataset: (590540, 327)


In [ ]:
# split as in the part 3 LGBM
n_samples = len(df)
split_idx = int(n_samples * 0.8)

idx_tr_time = np.arange(0, split_idx)
idx_val_time = np.arange(split_idx, n_samples)

Prepare data for NN - impute NaNs, scale features

In [ ]:
te_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
te_cols = [c for c in df.columns if c.endswith('_target_enc')]
drop_cols = [
    TARGET, 
    'TransactionDT', 
    'Pseudo_UID',
    'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26' # sparse IP/proxy with >95% NaN
] + te_cols  # Drop target encoding columns, DLRM doesn't need them

features = [c for c in df.columns if c not in drop_cols]

cat_cols=['card1', 'card2', 'card3', 'card5',
                        'addr1', 'addr2',
                        'DeviceInfo',
                        'id_30', 'id_31', 'id_33', 'id_13', 'id_14', 'id_17', 'id_19', 'id_20', 'id_32', 'id_34'] + te_features

bin_cols=['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9', 
          'id_12', 'id_15', 'id_16', 'id_27', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 
          'DeviceType']

block_bin_flags = [c for c in ['block_2_present', 'block_3_present', 'block_4_present'] if c in df.columns]

freq_cols = [c for c in df.columns if c.endswith('_freq')] # apply log1p before scaling

special_excluded = set(drop_cols + cat_cols + bin_cols + block_bin_flags + freq_cols + ['uid_dt_diff'])
numeric_cols = [c for c in df.columns if c not in special_excluded]

In [ ]:
# 2. Fit Preprocessor on Training Split Only & Save State
preprocessor = DLRMPreprocessor(
    numeric_cols=numeric_cols,
    freq_cols=freq_cols,
    cat_cols=cat_cols,
    bin_cols=bin_cols,
    block_bin_flags=block_bin_flags,
    min_freq=50,         # Try 10 vs 50 as an experiment
    scaler_type='robust',# Or 'quantile'
    clip_val=5.0         # Protects dot products from heavy-tailed numeric outliers
)

In [12]:
X_train_raw = df.iloc[idx_tr_time]
X_val_raw   = df.iloc[idx_val_time]
y_train     = df.iloc[idx_tr_time][TARGET].values.astype(np.float32)
y_val       = df.iloc[idx_val_time][TARGET].values.astype(np.float32)

In [ ]:
X_tr_cont, X_tr_cat = preprocessor.fit_transform(X_train_raw)
X_va_cont, X_va_cat = preprocessor.transform(X_val_raw)

/tmp/ipykernel_2868/1327183276.py:62: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_tr['id_15'] = X_tr['id_15'].replace('Unknown', np.nan)
/tmp/ipykernel_2868/1327183276.py:63: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_va['id_15'] = X_va['id_15'].replace('Unknown', np.nan)
/tmp/ipykernel_2868/1327183276.py:147: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downca

In [14]:
# 3. Assert zero NaNs and verify shapes
assert not np.isnan(X_tr_cont).any(), "NaN found in X_tr_cont!"
assert not np.isnan(X_va_cont).any(), "NaN found in X_va_cont!"
assert not np.isnan(X_tr_cat).any(),  "NaN found in X_tr_cat!"
assert not np.isnan(X_va_cat).any(),  "NaN found in X_va_cat!"

print(f"X_tr_cont shape: {X_tr_cont.shape} (dtype: {X_tr_cont.dtype})")
print(f"X_tr_cat  shape: {X_tr_cat.shape}  (dtype: {X_tr_cat.dtype})")
print(f"Total categorical embedding tables: {len(emb_szs)}")

X_tr_cont shape: (472432, 318) (dtype: float32)
X_tr_cat  shape: (472432, 17)  (dtype: int64)
Total categorical embedding tables: 17


In [ ]:
artifact_path=os.path.join(os.path.join(SAVE_DIR, "src"),"artifacts")
preprocessor.save(os.path.join(artifact_path, "dlrm_preprocessor_train.joblib"))  

In [ ]:
print(f"X_tr_cont shape: {X_tr_cont.shape} | X_tr_cat shape: {X_tr_cat.shape}")
print(f"Number of categorical tables: {len(preprocessor.cardinalities)}")

In [ ]:
# 3. Train Seed-Averaged DLRM with PR-AUC Early Stopping & MLflow Logging
mlflow.set_experiment("IEEE-fraud-DLRM")

dlrm_models, oof_val_preds = train_dlrm_seed_ensemble(
    X_tr_cont=X_tr_cont,
    X_tr_cat=X_tr_cat,
    y_train=y_train,
    X_va_cont=X_va_cont,
    X_va_cat=X_va_cat,
    y_val=y_val,
    cardinalities=preprocessor.cardinalities,
    save_dir=SAVE_DIR,
    run_name="dlrm_d16_minfreq50",
    seeds=[42, 100, 2026],
    d=16,
    batch_size=2048,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=20,
    patience=4,
    device="cpu"
)